In [ ]:
import cv2
import numpy as np
import tensorflow as tf
from pathlib import Path
from collections import deque

PROJECT_ROOT = Path("../..").resolve()
MODEL_PATH = PROJECT_ROOT / "experiments" / "06_cross_view_mobilenetv2" / "best_mobilenetv2_crossview_finetuned.keras"
IMG_SIZE = (224, 224)
LABEL_NAMES = [
    "safe_driving", "texting_right", "phone_right", "texting_left", "phone_left",
    "adjusting_radio", "drinking", "reaching_behind", "hair_or_makeup", "talking_to_passenger",
]

WEBCAM_INDEX = 0  # change if this isn't your laptop's camera (e.g. 1, 2)
SMOOTHING_WINDOW = 15       # averages over ~0.5-1s of frames, stops label flicker
CONFIDENCE_THRESHOLD = 0.40  # below this, shown as "uncertain" rather than committing to a guess

model = tf.keras.models.load_model(MODEL_PATH)
print("Model loaded:", MODEL_PATH)

In [ ]:
def predict_frame_probs(frame_bgr):
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    resized = tf.image.resize(frame_rgb, IMG_SIZE)
    batch = tf.expand_dims(tf.cast(resized, tf.float32), axis=0)
    return model.predict(batch, verbose=0)[0]


prob_history = deque(maxlen=SMOOTHING_WINDOW)

cap = cv2.VideoCapture(WEBCAM_INDEX)
if not cap.isOpened():
    raise RuntimeError(f"Could not open webcam (index {WEBCAM_INDEX}). Check that a camera is connected and not in use by another app.")

print("Webcam open. Press 'q' in the video window to quit.")

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Failed to read a frame - stopping.")
            break

        probs = predict_frame_probs(frame)
        prob_history.append(probs)
        smoothed_probs = np.mean(prob_history, axis=0)

        class_id = int(np.argmax(smoothed_probs))
        confidence = float(smoothed_probs[class_id])

        if confidence >= CONFIDENCE_THRESHOLD:
            label = LABEL_NAMES[class_id]
            color = (0, 200, 0) if class_id == 0 else (0, 0, 255)  # BGR: green=safe, red=distracted
        else:
            label = "uncertain"
            color = (0, 200, 200)

        display_frame = frame.copy()
        cv2.putText(display_frame, f"{label} ({confidence:.0%})", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
        cv2.imshow("Laptop model - live webcam", display_frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
finally:
    cap.release()
    cv2.destroyAllWindows()
    print("Stopped.")